# **TFM: Detecció d'esdeveniments importants en partits de futbol a partir de les seves narracions**

**Autor:** Martí Mullor Rordíguez

**Institució:** Universitat Oberta de Catalunya  
**Tutor:** Josep Mª Carmona Leyva

**Data:** Juny 2026

---

### **Nom de l'script: 04_Late_Fusion**

Aquest script implementa la fusió tardana (Late Fusion / Decision-Level Fusion) combinant les probabilitats predites pels models de text (SVM Baseline i RoBERTa) amb un classificador entrenat sobre els embeddings acústics de Wav2Vec2.

Està compost per:

**0. Importacions**

**1. Configuració inicial i càrrega de dades**

1.1 Muntar Google Drive  
1.2 Carregar la configuració inicial  
1.3 Definició de rutes i extracció de paràmetres i dades  

**2. Entrenament del model d'àudio (Wav2Vec2 Embeddings)**

**3. Late Fusion Model 1: SVM Baseline (Text) + Àudio**

**4. Late Fusion Model 2: RoBERTa (Text) + Àudio**

---


## **0. Importacions**

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import joblib
from google.colab import drive
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score

## **1. Configuració inicial i càrrega de dades**

### **1.1. Muntar Google Drive**

In [ ]:
if not os.path.exists('/content/drive'):
    print("Muntant Google Drive...")
    drive.mount('/content/drive')

### **1.2. Carregar la configuració inicial**

In [ ]:
CONFIG_PATH = "/content/drive/MyDrive/TFM/TFM-Deteccio-Esdeveniments-Futbol/config.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

### **1.3. Definició de rutes i extracció de paràmetres i dades**

In [ ]:
paths = config["paths"]
seed = config["global_settings"]["random_seed"]
fusion_params = config["late_fusion"]

np.random.seed(seed)

print("Carregant els DataFrames amb els resultats de Text i Embeddings d'Àudio...")

# 1. Carregar dades orígens (per als vectors d'àudio)
df_train = pd.read_pickle(os.path.join(paths["processed_data"], "train_dataset.pkl"))
df_val = pd.read_pickle(os.path.join(paths["processed_data"], "val_dataset.pkl"))
df_test = pd.read_pickle(os.path.join(paths["processed_data"], "test_dataset.pkl"))

# 2. Carregar el Label Encoder utilitzat a RoBERTa
label_encoder = joblib.load(os.path.join(paths["models"], "label_encoder.pkl"))
classes = label_encoder.classes_
num_classes = len(classes)

# 3. Carregar prediccions de l'SVM
svm_val = pd.read_pickle(os.path.join(paths["results"], "svm_val_predictions.pkl"))
svm_test = pd.read_pickle(os.path.join(paths["results"], "svm_test_predictions.pkl"))

# 4. Carregar prediccions de RoBERTa
roberta_val = pd.read_pickle(os.path.join(paths["results"], "roberta_val_predictions.pkl"))
roberta_test = pd.read_pickle(os.path.join(paths["results"], "roberta_test_predictions.pkl"))

print(f"Dades carregades correctament. Classes a classificar ({num_classes}): {classes}")

## **2. Entrenament del model d'àudio (Wav2Vec2 Embeddings)**

In [ ]:
print("Preparant la matriu d'embeddings d'àudio")

X_audio_train = np.vstack(df_train['audio_vector'].values)
y_audio_train_str = df_train['label'].values
y_audio_train_encoded = label_encoder.transform(y_audio_train_str)

X_audio_val = np.vstack(df_val['audio_vector'].values)
y_audio_val_encoded = label_encoder.transform(df_val['label'].values)

X_audio_test = np.vstack(df_test['audio_vector'].values)
y_audio_test_encoded = label_encoder.transform(df_test['label'].values)

print("Entrenant el classificador per a la modalitat d'àudio...")

audio_cfg = fusion_params.get("audio_classifier", {})
audio_model = LogisticRegression(
    max_iter=audio_cfg.get("max_iter", 1000),
    class_weight=audio_cfg.get("class_weight", "balanced"),
    solver=audio_cfg.get("solver", "lbfgs"),
    random_state=seed
)

audio_model.fit(X_audio_train, y_audio_train_encoded)

# Extracció de probabilitats d'àudio
audio_probs_val = audio_model.predict_proba(X_audio_val)
audio_probs_test = audio_model.predict_proba(X_audio_test)

# Guardem el model d'àudio
joblib.dump(audio_model, os.path.join(paths["models"], "audio_classifier_model.pkl"))
print("Model d'àudio entrenat i guardat amb èxit.")

## **3. Late Fusion Model 1: SVM Baseline (Text) + Àudio**

In [ ]:
# Matrius de probabilitats de text (SVM)
svm_probs_val = np.array(svm_val['svm_probs'].tolist())
svm_probs_test = np.array(svm_test['svm_probs'].tolist())

def optimize_fusion_weights(probs_text, probs_audio, y_true_encoded, steps=101):

    best_w = 0.5
    best_score = -1.0

    weights = np.linspace(0, 1, steps)
    for w in weights:
        # P_fused = w * P_text + (1 - w) * P_audio
        fused_probs = w * probs_text + (1 - w) * probs_audio
        preds = np.argmax(fused_probs, axis=1)
        score = f1_score(y_true_encoded, preds, average='macro', zero_division=0)

        if score > best_score:
            best_score = score
            best_w = w

    return best_w, best_score

steps = fusion_params.get("grid_search_steps", 101)
best_w_svm, best_val_score_svm = optimize_fusion_weights(
    svm_probs_val, audio_probs_val, y_audio_val_encoded, steps=steps
)

print(f"Pes òptim trobat a Validació (SVM): w_text = {best_w_svm:.2f} | w_audio = {1-best_w_svm:.2f}")
print(f"Macro F1 a Validació amb fusió SVM: {best_val_score_svm:.4f}")

# Aplicació del pes trobat al conjunt de TEST
fused_probs_svm_test = best_w_svm * svm_probs_test + (1 - best_w_svm) * audio_probs_test
y_pred_svm_fusion_encoded = np.argmax(fused_probs_svm_test, axis=1)
y_pred_svm_fusion_labels = label_encoder.inverse_transform(y_pred_svm_fusion_encoded)

print("\n--- Avaluació Late Fusion (SVM + Àudio) en TEST ---")
print(classification_report(df_test['label'], y_pred_svm_fusion_labels, zero_division=0))

## **4. Late Fusion Model 2: RoBERTa (Text) + Àudio**

In [ ]:
print("\n=== INICI LATE FUSION: RoBERTa + AUDIO ===")

# Matrius de probabilitats de text (RoBERTa)
roberta_probs_val = np.array(roberta_val['roberta_probs'].tolist())
roberta_probs_test = np.array(roberta_test['roberta_probs'].tolist())

best_w_roberta, best_val_score_roberta = optimize_fusion_weights(
    roberta_probs_val, audio_probs_val, y_audio_val_encoded, steps=steps
)

print(f"Pes òptim trobat a Validació (RoBERTa): w_text = {best_w_roberta:.2f} | w_audio = {1-best_w_roberta:.2f}")
print(f"Macro F1 a Validació amb fusió RoBERTa: {best_val_score_roberta:.4f}")

# Aplicació del pes trobat al conjunt de TEST
fused_probs_roberta_test = best_w_roberta * roberta_probs_test + (1 - best_w_roberta) * audio_probs_test
y_pred_roberta_fusion_encoded = np.argmax(fused_probs_roberta_test, axis=1)
y_pred_roberta_fusion_labels = label_encoder.inverse_transform(y_pred_roberta_fusion_encoded)

print("\n--- Avaluació Late Fusion (RoBERTa + Àudio) en TEST ---")
print(classification_report(df_test['label'], y_pred_roberta_fusion_labels, zero_division=0))

## **5. Avaluació comparativa i exportació de resultats**

In [ ]:
# Prediccions individuals prèvies
svm_only_preds = svm_test['svm_pred']
roberta_only_preds = roberta_test['roberta_pred']
audio_only_preds = label_encoder.inverse_transform(np.argmax(audio_probs_test, axis=1))

y_true = df_test['label']

summary_df = pd.DataFrame([
    {"Model": "SVM Baseline (Només Text)", "Macro F1": f1_score(y_true, svm_only_preds, average='macro', zero_division=0), "Accuracy": accuracy_score(y_true, svm_only_preds)},
    {"Model": "RoBERTa (Només Text)", "Macro F1": f1_score(y_true, roberta_only_preds, average='macro', zero_division=0), "Accuracy": accuracy_score(y_true, roberta_only_preds)},
    {"Model": "Wav2Vec2 (Només Àudio)", "Macro F1": f1_score(y_true, audio_only_preds, average='macro', zero_division=0), "Accuracy": accuracy_score(y_true, audio_only_preds)},
    {"Model": "Late Fusion 1: SVM + Àudio", "Macro F1": f1_score(y_true, y_pred_svm_fusion_labels, average='macro', zero_division=0), "Accuracy": accuracy_score(y_true, y_pred_svm_fusion_labels)},
    {"Model": "Late Fusion 2: RoBERTa + Àudio", "Macro F1": f1_score(y_true, y_pred_roberta_fusion_labels, average='macro', zero_division=0), "Accuracy": accuracy_score(y_true, y_pred_roberta_fusion_labels)},
])

print(summary_df.to_string(index=False))

# Guardar DataFrames amb totes les prediccions
df_val_results = df_val.copy()
df_val_results['audio_probs'] = audio_probs_val.tolist()
df_val_results['late_fusion_svm_pred'] = label_encoder.inverse_transform(
    np.argmax(best_w_svm * svm_probs_val + (1 - best_w_svm) * audio_probs_val, axis=1)
)
df_val_results['late_fusion_roberta_pred'] = label_encoder.inverse_transform(
    np.argmax(best_w_roberta * roberta_probs_val + (1 - best_w_roberta) * audio_probs_val, axis=1)
)

df_test_results = df_test.copy()
df_test_results['audio_probs'] = audio_probs_test.tolist()
df_test_results['late_fusion_svm_pred'] = y_pred_svm_fusion_labels
df_test_results['late_fusion_roberta_pred'] = y_pred_roberta_fusion_labels

# Exportació de resultats
df_val_results.to_pickle(os.path.join(paths["results"], "late_fusion_val_predictions.pkl"))
df_test_results.to_pickle(os.path.join(paths["results"], "late_fusion_test_predictions.pkl"))

# Guardar weights de fusió
fusion_metadata = {
    "weights": {
        "svm_text_weight": float(best_w_svm),
        "svm_audio_weight": float(1 - best_w_svm),
        "roberta_text_weight": float(best_w_roberta),
        "roberta_audio_weight": float(1 - best_w_roberta)
    }
}

with open(os.path.join(paths["models"], "late_fusion_weights.json"), "w") as f:
    json.dump(fusion_metadata, f, indent=4)

print("\n Arxius exportats correctament a RESULTS_PATH i MODELS_PATH.")